# SpatialData I/O with Segger

This notebook demonstrates how to:
1. Read spatial transcriptomics data from SpatialData Zarr stores
2. Run Segger segmentation (simulated)
3. Export results to SpatialData-compatible Zarr format
4. Validate compatibility with SOPA workflows

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import polars as pl
import geopandas as gpd
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

## 1. Create Sample Data

First, let's create some synthetic Xenium-like data to work with.

In [ ]:
from segger.datasets import create_synthetic_xenium

# Create synthetic Xenium data
transcripts, cells, boundaries = create_synthetic_xenium(
    n_cells=100,
    transcripts_per_cell=30,
    seed=42,
)

print(f"Generated {len(transcripts):,} transcripts")
print(f"Generated {len(cells):,} cells")
print(f"Generated {len(boundaries):,} cell boundaries")
print(f"\nTranscript columns: {transcripts.columns}")

In [ ]:
# Visualize the data
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot transcripts
ax1 = axes[0]
tx_pd = transcripts.to_pandas()
ax1.scatter(tx_pd['x_location'], tx_pd['y_location'], s=1, alpha=0.5)
ax1.set_xlabel('X (microns)')
ax1.set_ylabel('Y (microns)')
ax1.set_title('Transcript Positions')
ax1.set_aspect('equal')

# Plot boundaries
ax2 = axes[1]
boundaries.plot(ax=ax2, facecolor='lightblue', edgecolor='navy', alpha=0.5)
ax2.set_xlabel('X (microns)')
ax2.set_ylabel('Y (microns)')
ax2.set_title('Cell Boundaries')
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

## 2. Write to SpatialData Zarr Format

Segger includes a lightweight SpatialData writer that creates Zarr stores compatible with the scverse ecosystem and SOPA.

In [ ]:
from segger.io.spatialdata_zarr import (
    SpatialDataZarrWriter,
    SpatialDataZarrReader,
    write_spatialdata_zarr,
    read_spatialdata_zarr,
    get_spatialdata_info,
)

# Create a temporary directory for output
output_dir = Path(tempfile.mkdtemp())
zarr_path = output_dir / "experiment.zarr"

# Standardize column names
tx_standard = transcripts.rename({
    'x_location': 'x',
    'y_location': 'y',
    'z_location': 'z',
})

# Write to SpatialData format
write_spatialdata_zarr(
    tx_standard,
    zarr_path,
    shapes=boundaries,
    points_key="transcripts",
    shapes_key="cells",
)

print(f"Wrote SpatialData to: {zarr_path}")
print(f"\nStore info: {get_spatialdata_info(zarr_path)}")

## 3. Read from SpatialData Zarr Format

The reader can load data from any SpatialData-compatible Zarr store.

In [ ]:
# Using the class-based reader
reader = SpatialDataZarrReader(zarr_path)

print("Available elements:")
print(f"  Points: {reader.points_keys}")
print(f"  Shapes: {reader.shapes_keys}")

# Read data
tx_loaded = reader.read_points("transcripts")
shapes_loaded = reader.read_shapes("cells")

print(f"\nLoaded {len(tx_loaded):,} transcripts")
print(f"Loaded {len(shapes_loaded):,} cell boundaries")

In [ ]:
# Or use the convenience function
tx_loaded, shapes_loaded = read_spatialdata_zarr(zarr_path)

print(f"Loaded transcripts shape: {tx_loaded.shape}")
print(f"Loaded shapes shape: {shapes_loaded.shape}")

## 4. Simulate Segger Segmentation

Now let's simulate a Segger segmentation run. In practice, you would run:
```bash
segger segment -i data/ -o output/
```

Here we'll use the sample output generator.

In [ ]:
from segger.datasets import create_sample_segger_output, create_merged_output

# Generate sample Segger outputs
tx_data, predictions, cell_boundaries = create_sample_segger_output(
    n_cells=100,
    transcripts_per_cell=30,
    unassigned_rate=0.1,  # 10% unassigned
    seed=42,
)

print("Segger Predictions Format:")
print(predictions.head())

# Statistics
n_assigned = (predictions['segger_cell_id'] >= 0).sum()
n_unassigned = (predictions['segger_cell_id'] < 0).sum()
print(f"\nAssigned: {n_assigned:,} ({100*n_assigned/len(predictions):.1f}%)")
print(f"Unassigned: {n_unassigned:,} ({100*n_unassigned/len(predictions):.1f}%)")

In [ ]:
# Create merged output (transcripts + predictions)
merged = create_merged_output(tx_data, predictions)

print("Merged Output Format:")
print(merged.head())
print(f"\nColumns: {merged.columns}")

## 5. Export Segger Results to SpatialData

Convert the Segger segmentation results to SpatialData format for use with SOPA and other scverse tools.

In [ ]:
# Export to SpatialData
segmentation_zarr = output_dir / "segmentation.zarr"

write_spatialdata_zarr(
    merged,
    segmentation_zarr,
    shapes=cell_boundaries,
    points_key="transcripts",
    shapes_key="cells",
)

print(f"Exported segmentation to: {segmentation_zarr}")
print(f"\nStore info: {get_spatialdata_info(segmentation_zarr)}")

In [ ]:
# Verify the export
reader = SpatialDataZarrReader(segmentation_zarr)
tx_exported = reader.read_points()

print("Exported transcripts columns:")
print(tx_exported.columns)

# Check segmentation columns are present
assert 'segger_cell_id' in tx_exported.columns
assert 'segger_similarity' in tx_exported.columns
print("\n✓ Segmentation columns present in exported SpatialData")

## 6. Visualize Segmentation Results

In [ ]:
# Visualize assigned vs unassigned transcripts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

merged_pd = merged.to_pandas()
assigned = merged_pd[merged_pd['segger_cell_id'] >= 0]
unassigned = merged_pd[merged_pd['segger_cell_id'] < 0]

# Left: All transcripts colored by assignment
ax1 = axes[0]
ax1.scatter(assigned['x'], assigned['y'], s=2, c='blue', alpha=0.5, label='Assigned')
ax1.scatter(unassigned['x'], unassigned['y'], s=5, c='red', alpha=0.8, label='Unassigned')
ax1.set_xlabel('X (microns)')
ax1.set_ylabel('Y (microns)')
ax1.set_title('Transcript Assignment Status')
ax1.legend()
ax1.set_aspect('equal')

# Right: Transcripts colored by cell ID
ax2 = axes[1]
scatter = ax2.scatter(
    assigned['x'], assigned['y'], 
    s=2, 
    c=assigned['segger_cell_id'], 
    cmap='tab20',
    alpha=0.7
)
ax2.set_xlabel('X (microns)')
ax2.set_ylabel('Y (microns)')
ax2.set_title('Transcripts by Cell ID')
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize similarity scores
fig, ax = plt.subplots(figsize=(10, 4))

similarities = merged_pd[merged_pd['segger_similarity'] > 0]['segger_similarity']
ax.hist(similarities, bins=50, edgecolor='white', alpha=0.7)
ax.axvline(similarities.median(), color='red', linestyle='--', label=f'Median: {similarities.median():.3f}')
ax.set_xlabel('Similarity Score')
ax.set_ylabel('Count')
ax.set_title('Distribution of Assignment Similarity Scores')
ax.legend()

plt.tight_layout()
plt.show()

## 7. Save All Outputs

Use the `save_sample_outputs` function to generate a complete set of sample files.

In [ ]:
from segger.datasets import save_sample_outputs

sample_output_dir = output_dir / "sample_outputs"
paths = save_sample_outputs(
    sample_output_dir,
    n_cells=50,
    transcripts_per_cell=20,
    include_spatialdata=True,
)

print("Generated sample outputs:")
for name, path in paths.items():
    print(f"  {name}: {path}")

## 8. SOPA Compatibility

The exported SpatialData Zarr stores are compatible with SOPA workflows. Key conventions:

- **shapes["cells"]**: Cell polygons with `cell_id` column
- **points["transcripts"]**: Transcripts with `segger_cell_id` assignment column
- Coordinate systems use identity transforms

To use with SOPA:
```python
import sopa
import spatialdata

# Load Segger output
sdata = spatialdata.read_zarr("segmentation.zarr")

# Continue with SOPA analysis
sopa.aggregate(sdata, ...)
```

In [ ]:
# Cleanup
import shutil
shutil.rmtree(output_dir)
print("Cleaned up temporary files")

## Summary

This notebook demonstrated:

1. **Creating synthetic data** with `create_synthetic_xenium()`
2. **Writing to SpatialData** with `write_spatialdata_zarr()`
3. **Reading from SpatialData** with `read_spatialdata_zarr()` or `SpatialDataZarrReader`
4. **Simulating Segger output** with `create_sample_segger_output()`
5. **Exporting segmentation results** to SpatialData-compatible Zarr stores
6. **SOPA compatibility** for downstream analysis

### Key Functions

| Function | Purpose |
|----------|------|
| `write_spatialdata_zarr()` | Write transcripts + shapes to Zarr |
| `read_spatialdata_zarr()` | Read transcripts + shapes from Zarr |
| `SpatialDataZarrReader` | Class-based reader with metadata |
| `SpatialDataZarrWriter` | Class-based writer with incremental writes |
| `create_sample_segger_output()` | Generate test predictions |
| `save_sample_outputs()` | Save complete sample dataset |